## Step 1: Install Dependencies & Setup

In [ ]:
# Install dependencies - FIXED VERSION

# Fix torch version conflicts first
!pip install torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 -q --index-url https://download.pytorch.org/whl/cu121

# Install main dependencies
!pip install -q diffusers[torch]==0.31.0 transformers==4.44.2 accelerate==0.34.2 peft==0.13.2
!pip install -q xformers==0.0.28.post2 --index-url https://download.pytorch.org/whl/cu121
!pip install -q opencv-python pandas openpyxl tqdm bitsandbytes scikit-learn

print("Dependencies installed (torch versions aligned)")

In [ ]:
import torch
import os

# Verify A100 GPU
assert torch.cuda.is_available(),"No GPU detected!"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

if"A100"not in gpu_name:
 print("WARNING: Not using A100. Optimizations may need adjustment.")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Configuration Selection

**Choose which configuration to train (1-7)**

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ========== CHANGE THIS TO SELECT CONFIGURATION ==========
CONFIG_ID = 7 # 1=Canny, 2=Seg, 3=Depth, 4=Canny+Seg, 5=Canny+Depth, 6=Seg+Depth, 7=Triple
# =========================================================

CONFIGURATIONS = {
 1: {"name":"Canny","controls": ["canny"],"dirs": ["CannyMap"],
"models": ["lllyasviel/control_v11p_sd15_canny"]},
 2: {"name":"Segmentation","controls": ["seg"],"dirs": ["SegmentationMap"],
"models": ["lllyasviel/control_v11p_sd15_seg"]},
 3: {"name":"Depth","controls": ["depth"],"dirs": ["DepthMap"],
"models": ["lllyasviel/control_v11f1p_sd15_depth"]},
 4: {"name":"Canny+Seg","controls": ["canny","seg"],"dirs": ["CannyMap","SegmentationMap"],
"models": ["lllyasviel/control_v11p_sd15_canny","lllyasviel/control_v11p_sd15_seg"]},
 5: {"name":"Canny+Depth","controls": ["canny","depth"],"dirs": ["CannyMap","DepthMap"],
"models": ["lllyasviel/control_v11p_sd15_canny","lllyasviel/control_v11f1p_sd15_depth"]},
 6: {"name":"Seg+Depth","controls": ["seg","depth"],"dirs": ["SegmentationMap","DepthMap"],
"models": ["lllyasviel/control_v11p_sd15_seg","lllyasviel/control_v11f1p_sd15_depth"]},
 7: {"name":"Triple","controls": ["canny","seg","depth"],
"dirs": ["CannyMap","SegmentationMap","DepthMap"],
"models": ["lllyasviel/control_v11p_sd15_canny","lllyasviel/control_v11p_sd15_seg",
"lllyasviel/control_v11f1p_sd15_depth"]},
}

config = CONFIGURATIONS[CONFIG_ID]
print(f"\n Selected Configuration: {config['name']}")
print(f"Controls: {', '.join(config['controls'])}")
print(f"Number of ControlNets: {len(config['controls'])}")

## Step 3: Path Configuration

In [ ]:
BASE_PATH ="/content/drive/MyDrive/5190 S26 Final Project"


# Input paths
DATASET_PATH = f"{BASE_PATH}/Dataset/MODELIMG"
IMAGE_DIR = f"{DATASET_PATH}/Resized and cleaned"
METADATA_PATH = f"{BASE_PATH}/Dataset/BoundaryMetadata.xlsx"

# Control map directories
CONTROL_PATHS = {name: f"{BASE_PATH}/Dataset/MODELIMG/{dir_name}"
 for name, dir_name in zip(config['controls'], config['dirs'])}

# Output directory for this configuration
OUTPUT_DIR = f"{BASE_PATH}/Training_Outputs/{config['name']}"
LORA_SAVE_DIR = f"{OUTPUT_DIR}/lora_checkpoints"
VALIDATION_DIR = f"{OUTPUT_DIR}/validation_outputs"

os.makedirs(LORA_SAVE_DIR, exist_ok=True)
os.makedirs(VALIDATION_DIR, exist_ok=True)

print(f"\n Paths configured:")
print(f"Images: {IMAGE_DIR}")
print(f"Metadata: {METADATA_PATH}")
for name, path in CONTROL_PATHS.items():
 print(f"{name}: {path}")
print(f"Output: {OUTPUT_DIR}")

## Step 4: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


from diffusers import (
 StableDiffusionControlNetPipeline,
 ControlNetModel,
 DDPMScheduler,
 AutoencoderKL,
 UNet2DConditionModel,
)
from transformers import CLIPTextModel, CLIPTokenizer
from peft import LoraConfig, get_peft_model, PeftModel

device ="cuda"
dtype = torch.float16

print("Libraries imported")

## Step 5: Load Metadata & Create Train/Val Split

In [ ]:
# Load metadata from Excel
metadata_df = pd.read_excel(METADATA_PATH, sheet_name='Prompts and MD')

print(f"Total images in metadata: {len(metadata_df)}")
print(f"\nColumns: {list(metadata_df.columns)}")

# Train/Val split (80/20)
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
 metadata_df,
 test_size=0.2,
 random_state=42,
 stratify=metadata_df['time_of_day'] # Stratify by time to keep balanced
)

print(f"\nTrain samples: {len(train_df)}")
print(f"Val samples: {len(val_df)}")

# Sample prompts
print(f"\nSample training data:")
sample = train_df.iloc[0]
print(f"Filename: {sample['filename']}")
print(f"Prompt: {sample['prompt'][:100]}...")
print(f"Negative: {sample['negative_prompt'][:80]}...")

## Step 6: Dataset Class with Negative Prompts

In [ ]:
class PennStyleTransferDataset(Dataset):
"""
 A100-optimized dataset with:
 - Dynamic control map loading (1-3 ControlNets)
 - Efficient transforms
 - Non-blocking transfers ready
"""

 def __init__(self, df, image_dir, control_paths, tokenizer, resolution=512):
 self.df = df.reset_index(drop=True)
 self.image_dir = image_dir
 self.control_paths = control_paths # Dict: {control_name: path}
 self.tokenizer = tokenizer
 self.resolution = resolution

 # Validate files exist
 self._validate_files()

 # Image transform (for VAE): normalize to [-1, 1]
 self.image_transform = transforms.Compose([
 transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BILINEAR),
 transforms.CenterCrop(resolution),
 transforms.ToTensor(),
 transforms.Normalize([0.5], [0.5]),
 ])

 # Control transform (for ControlNet): normalize to [0, 1]
 self.control_transform = transforms.Compose([
 transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BILINEAR),
 transforms.CenterCrop(resolution),
 transforms.ToTensor(),
 ])

 def _validate_files(self):
"""Remove samples with missing files"""
 valid_indices = []

 for idx, row in self.df.iterrows():
 filename = row['filename']

 # Check image exists
 img_path = os.path.join(self.image_dir, filename)
 if not os.path.exists(img_path):
 continue

 # Check all control maps exist
 all_exist = True
 for control_path in self.control_paths.values():
 if not os.path.exists(os.path.join(control_path, filename)):
 all_exist = False
 break

 if all_exist:
 valid_indices.append(idx)

 initial_count = len(self.df)
 self.df = self.df.loc[valid_indices].reset_index(drop=True)

 print(f"Dataset validation: {len(self.df)}/{initial_count} samples have all required files")

 def __len__(self):
 return len(self.df)

 def __getitem__(self, idx):
 row = self.df.iloc[idx]
 filename = row['filename']
 prompt = row['prompt']

 # Load image
 img_path = os.path.join(self.image_dir, filename)
 image = Image.open(img_path).convert("RGB")
 pixel_values = self.image_transform(image)

 # Load control maps (1-3 depending on config)
 control_images = []
 for control_path in self.control_paths.values():
 control_img_path = os.path.join(control_path, filename)
 control_img = Image.open(control_img_path).convert("RGB")
 control_tensor = self.control_transform(control_img)
 control_images.append(control_tensor)

 # Tokenize prompt only (negative prompts handled via CFG dropout in training)
 input_ids = self.tokenizer(
 prompt,
 padding="max_length",
 max_length=self.tokenizer.model_max_length,
 truncation=True,
 return_tensors="pt"
 ).input_ids[0]

 return {
"pixel_values": pixel_values,
"conditioning_images": control_images, # List of 1-3 tensors
"input_ids": input_ids,
 }

print("Dataset class defined (CFG-ready)")


## Step 7: Load Models (SD + ControlNets)

In [ ]:
print("Loading models...\n")

# Load tokenizer
tokenizer = CLIPTokenizer.from_pretrained(
"runwayml/stable-diffusion-v1-5",
 subfolder="tokenizer"
)
print("Tokenizer loaded")

# Load text encoder
text_encoder = CLIPTextModel.from_pretrained(
"runwayml/stable-diffusion-v1-5",
 subfolder="text_encoder",
 torch_dtype=dtype
).to(device)
text_encoder.requires_grad_(False) # Freeze
print("Text encoder loaded (frozen)")

# Load VAE
vae = AutoencoderKL.from_pretrained(
"runwayml/stable-diffusion-v1-5",
 subfolder="vae",
 torch_dtype=dtype
).to(device)
vae.requires_grad_(False) # Freeze
print("VAE loaded (frozen)")

# Load ControlNets (1-3 depending on config)
controlnets = []
for model_id in config['models']:
 cn = ControlNetModel.from_pretrained(
 model_id,
 torch_dtype=dtype
 ).to(device)
 cn.requires_grad_(False) # Freeze ControlNets
 controlnets.append(cn)
 print(f"ControlNet loaded: {model_id.split('/')[-1]} (frozen)")

# Load UNet with LoRA
unet = UNet2DConditionModel.from_pretrained(
"runwayml/stable-diffusion-v1-5",
 subfolder="unet",
 torch_dtype=dtype
).to(device)
print("UNet loaded")

# Enable gradient checkpointing for memory efficiency
unet.enable_gradient_checkpointing()
print("Gradient checkpointing enabled")

# Enable xformers for memory-efficient attention
try:
 unet.enable_xformers_memory_efficient_attention()
 print("XFormers enabled")
except Exception as e:
 print(f"XFormers not available: {e}")

print(f"\n Model summary:")
print(f"ControlNets: {len(controlnets)}")
print(f"Total trainable params: {sum(p.numel() for p in unet.parameters() if p.requires_grad):,}")

## Step 8: Apply LoRA to UNet

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
 r=32, # Rank
 lora_alpha=32, # Scaling
 target_modules=["to_q","to_k","to_v","to_out.0"], # Attention layers
 lora_dropout=0.1,
 bias="none",
)

# Apply LoRA
unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

print("\n LoRA applied to UNet")

## Step 9: Create Datasets & DataLoaders (A100 Optimized)

In [ ]:
# Create datasets
train_dataset = PennStyleTransferDataset(
 df=train_df,
 image_dir=IMAGE_DIR,
 control_paths=CONTROL_PATHS,
 tokenizer=tokenizer,
 resolution=512
)

val_dataset = PennStyleTransferDataset(
 df=val_df,
 image_dir=IMAGE_DIR,
 control_paths=CONTROL_PATHS,
 tokenizer=tokenizer,
 resolution=512
)

print(f"\nDatasets created:")
print(f"Train: {len(train_dataset)} samples")
print(f"Val: {len(val_dataset)} samples")

# Collate function for batching
def collate_fn(examples):
 pixel_values = torch.stack([example["pixel_values"] for example in examples])
 input_ids = torch.stack([example["input_ids"] for example in examples])

 # Stack control images (handles 1-3 ControlNets)
 num_controls = len(examples[0]["conditioning_images"])
 conditioning_images = [
 torch.stack([example["conditioning_images"][i] for example in examples])
 for i in range(num_controls)
 ]

 return {
"pixel_values": pixel_values,
"conditioning_images": conditioning_images,
"input_ids": input_ids,
 }

# A100 optimized DataLoader settings
train_dataloader = DataLoader(
 train_dataset,
 batch_size=4, # A100 can handle this
 shuffle=True,
 num_workers=16, # Max CPU utilization
 pin_memory=True, # Faster GPU transfer
 persistent_workers=True, # Keep workers alive
 prefetch_factor=4, # Prefetch 4 batches per worker
 collate_fn=collate_fn,
)

val_dataloader = DataLoader(
 val_dataset,
 batch_size=4,
 shuffle=False,
 num_workers=8, # Fewer for validation
 pin_memory=True,
 collate_fn=collate_fn,
)

print(f"\n DataLoaders created (A100 optimized):")
print(f"Batch size: 4")
print(f"Train workers: 16")
print(f"Val workers: 8")
print(f"Prefetch factor: 4")


In [ ]:
# Find which images are missing control maps

import pandas as pd
import os


# Load metadata
metadata_df = pd.read_excel(METADATA_PATH, sheet_name='Metadata')

# Check which files are missing
missing_files = []

for filename in metadata_df['filename']:
 img_path = os.path.join(IMAGE_DIR, filename)
 canny_path = f"{BASE_PATH}/Dataset/MODELIMG/CannyMap/{filename}"
 depth_path = f"{BASE_PATH}/Dataset/MODELIMG/DepthMap/{filename}"
 seg_path = f"{BASE_PATH}/Dataset/MODELIMG/SegmentationMap/{filename}"

 # Check if all exist
 has_img = os.path.exists(img_path)
 has_canny = os.path.exists(canny_path)
 has_depth = os.path.exists(depth_path)
 has_seg = os.path.exists(seg_path)

 if not (has_img and has_canny and has_depth and has_seg):
 missing_files.append({
 'filename': filename,
 'image': has_img,
 'canny': has_canny,
 'depth': has_depth,
 'segmentation': has_seg
 })

# Print results
print(f"Missing files: {len(missing_files)} / {len(metadata_df)}\n")

if missing_files:
 missing_df = pd.DataFrame(missing_files)
 print(missing_df.to_string())
else:
 print("All files present!")

## Step 10: Training Configuration

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

# A100-optimized training hyperparameters (tuned for LPIPS quality)
num_epochs = 25 # More epochs with early stopping
learning_rate = 1e-4 # Higher LR works better with LoRA + warmup
adam_beta1 = 0.9
adam_beta2 = 0.999
adam_weight_decay = 0.01
adam_epsilon = 1e-8
max_grad_norm = 1.0

# Conditional dropout for proper CFG training
conditional_dropout_prob = 0.1 # 10% chance to drop text conditioning

# Optimizer (A100 optimized with fused implementation, only LoRA params)
optimizer = AdamW(
 [p for p in unet.parameters() if p.requires_grad], # Only LoRA params
 lr=learning_rate,
 betas=(adam_beta1, adam_beta2),
 weight_decay=adam_weight_decay,
 eps=adam_epsilon,
 fused=True # A100 optimization - faster on Ampere GPUs
)

# OneCycleLR with proper warmup
total_steps = num_epochs * len(train_dataloader)
scheduler = OneCycleLR(
 optimizer,
 max_lr=learning_rate,
 total_steps=total_steps,
 pct_start=0.1, # 10% warmup
 anneal_strategy='cos',
 div_factor=25.0, # Initial lr = max_lr / 25
 final_div_factor=1000.0 # Final lr = max_lr / 1000
)

# Noise scheduler
noise_scheduler = DDPMScheduler.from_pretrained(
"runwayml/stable-diffusion-v1-5",
 subfolder="scheduler"
)

# Get unconditional embeddings (empty string) for CFG dropout
with torch.no_grad():
 uncond_input = tokenizer(
 [""],
 padding="max_length",
 max_length=tokenizer.model_max_length,
 truncation=True,
 return_tensors="pt"
 )
 uncond_embeddings = text_encoder(uncond_input.input_ids.to(device))[0]

print(f"A100-Optimized Training Configuration:")
print(f"Max epochs: {num_epochs} (early stopping enabled)")
print(f"Learning rate: {learning_rate} (OneCycleLR)")
print(f"Optimizer: AdamW (fused=True for A100)")
print(f"Scheduler: OneCycleLR with cosine annealing")
print(f"Conditional dropout: {conditional_dropout_prob} (proper CFG training)")
print(f"Trainable params: {sum(p.numel() for p in unet.parameters() if p.requires_grad):,}")
print(f"Total training steps: {total_steps}")


## Step 11: Training Loop with Negative Prompts

In [ ]:
from torch.cuda.amp import autocast, GradScaler
import time

# Mixed precision scaler
scaler = GradScaler()

# Early stopping variables
best_val_loss = float('inf')
patience = 5
patience_counter = 0
min_delta = 0.0001


def compute_snr(timesteps, noise_scheduler):
"""Compute Signal-to-Noise Ratio for SNR-weighted loss"""
 alphas_cumprod = noise_scheduler.alphas_cumprod.to(timesteps.device)
 sqrt_alphas_cumprod = alphas_cumprod[timesteps] ** 0.5
 sqrt_one_minus_alphas_cumprod = (1.0 - alphas_cumprod[timesteps]) ** 0.5
 snr = (sqrt_alphas_cumprod / sqrt_one_minus_alphas_cumprod) ** 2
 return snr


def train_epoch(epoch):
 unet.train()
 total_loss = 0

 progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")

 for step, batch in enumerate(progress_bar):
 # Move to device with non_blocking for faster transfer
 pixel_values = batch["pixel_values"].to(device, dtype=dtype, non_blocking=True)
 conditioning_images = [
 img.to(device, dtype=dtype, non_blocking=True)
 for img in batch["conditioning_images"]
 ]
 input_ids = batch["input_ids"].to(device, non_blocking=True)
 bsz = pixel_values.shape[0]

 with autocast(dtype=dtype):
 # Encode images to latents (frozen VAE - no gradient needed)
 with torch.no_grad():
 latents = vae.encode(pixel_values).latent_dist.sample()
 latents = latents * vae.config.scaling_factor

 # Sample noise
 noise = torch.randn_like(latents)

 # Sample timesteps
 timesteps = torch.randint(
 0, noise_scheduler.config.num_train_timesteps, (bsz,),
 device=device
 ).long()

 # Add noise to latents
 noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

 # Get text embeddings (frozen text encoder - no gradient needed)
 with torch.no_grad():
 encoder_hidden_states = text_encoder(input_ids)[0]

 # PROPER CFG TRAINING: Randomly drop conditioning for ~10% of samples
 # This trains model to handle both conditional and unconditional generation
 random_mask = torch.rand(bsz, device=device) < conditional_dropout_prob
 if random_mask.any():
 encoder_hidden_states = torch.where(
 random_mask[:, None, None],
 uncond_embeddings.expand(bsz, -1, -1),
 encoder_hidden_states
 )

 # Apply ControlNets (frozen - no gradient needed)
 down_block_res_samples_list = []
 mid_block_res_sample_list = []

 for controlnet, cond_img in zip(controlnets, conditioning_images):
 with torch.no_grad():
 down_block_res_samples, mid_block_res_sample = controlnet(
 noisy_latents,
 timesteps,
 encoder_hidden_states=encoder_hidden_states,
 controlnet_cond=cond_img,
 return_dict=False,
 )
 down_block_res_samples_list.append(down_block_res_samples)
 mid_block_res_sample_list.append(mid_block_res_sample)

 # Combine ControlNet outputs (averaged, as Diffusers does)
 if len(controlnets) > 1:
 down_block_res_samples = tuple(
 sum([d[i] for d in down_block_res_samples_list]) / len(controlnets)
 for i in range(len(down_block_res_samples_list[0]))
 )
 mid_block_res_sample = sum(mid_block_res_sample_list) / len(controlnets)
 else:
 down_block_res_samples = down_block_res_samples_list[0]
 mid_block_res_sample = mid_block_res_sample_list[0]

 # Single forward pass through UNet (only this has gradients)
 noise_pred = unet(
 noisy_latents,
 timesteps,
 encoder_hidden_states=encoder_hidden_states,
 down_block_additional_residuals=down_block_res_samples,
 mid_block_additional_residual=mid_block_res_sample,
 ).sample

 # SNR-weighted MSE loss (better for LPIPS quality)
 # Min-SNR weighting prevents overfitting on easy timesteps
 snr = compute_snr(timesteps, noise_scheduler)
 mse_loss_weights = torch.stack(
 [snr, 5.0 * torch.ones_like(timesteps)], dim=1
 ).min(dim=1)[0] / snr

 loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="none")
 loss = loss.mean(dim=list(range(1, len(loss.shape))))
 loss = (loss * mse_loss_weights).mean()

 # Backward pass
 optimizer.zero_grad()
 scaler.scale(loss).backward()

 # Gradient clipping (only on LoRA params - faster)
 scaler.unscale_(optimizer)
 torch.nn.utils.clip_grad_norm_(
 [p for p in unet.parameters() if p.requires_grad],
 max_grad_norm
 )

 scaler.step(optimizer)
 scaler.update()
 scheduler.step()

 # Log
 total_loss += loss.detach().item()
 progress_bar.set_postfix({
"loss": f"{loss.item():.4f}",
"lr": f"{optimizer.param_groups[0]['lr']:.2e}"
 })

 avg_loss = total_loss / len(train_dataloader)
 return avg_loss


print("Training function defined (with proper CFG + SNR weighting)")


## Step 12: Validation Function

In [ ]:
@torch.no_grad()
def validate():
 unet.eval()
 total_loss = 0

 for batch in tqdm(val_dataloader, desc="Validation", leave=False):
 pixel_values = batch["pixel_values"].to(device, dtype=dtype, non_blocking=True)
 conditioning_images = [
 img.to(device, dtype=dtype, non_blocking=True)
 for img in batch["conditioning_images"]
 ]
 input_ids = batch["input_ids"].to(device, non_blocking=True)
 bsz = pixel_values.shape[0]

 with autocast(dtype=dtype):
 # Encode
 latents = vae.encode(pixel_values).latent_dist.sample()
 latents = latents * vae.config.scaling_factor

 # Noise
 noise = torch.randn_like(latents)
 timesteps = torch.randint(
 0, noise_scheduler.config.num_train_timesteps, (bsz,),
 device=device
 ).long()

 noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
 encoder_hidden_states = text_encoder(input_ids)[0]

 # ControlNet
 down_block_res_samples_list = []
 mid_block_res_sample_list = []

 for controlnet, cond_img in zip(controlnets, conditioning_images):
 down, mid = controlnet(
 noisy_latents, timesteps,
 encoder_hidden_states=encoder_hidden_states,
 controlnet_cond=cond_img,
 return_dict=False,
 )
 down_block_res_samples_list.append(down)
 mid_block_res_sample_list.append(mid)

 # Average if multiple
 if len(controlnets) > 1:
 down_block_res_samples = tuple(
 sum([d[i] for d in down_block_res_samples_list]) / len(controlnets)
 for i in range(len(down_block_res_samples_list[0]))
 )
 mid_block_res_sample = sum(mid_block_res_sample_list) / len(controlnets)
 else:
 down_block_res_samples = down_block_res_samples_list[0]
 mid_block_res_sample = mid_block_res_sample_list[0]

 # Predict
 noise_pred = unet(
 noisy_latents, timesteps,
 encoder_hidden_states=encoder_hidden_states,
 down_block_additional_residuals=down_block_res_samples,
 mid_block_additional_residual=mid_block_res_sample,
 ).sample

 loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="mean")
 total_loss += loss.item()

 avg_loss = total_loss / len(val_dataloader)
 return avg_loss

print("Validation function defined")


## Step 13: Run Training

In [ ]:
import time

# Early stopping configuration
patience = 5
patience_counter = 0
min_delta = 0.0001
best_val_loss = float('inf')


def save_lora_weights(unet_model, save_path):
"""Save only LoRA weights (not entire UNet)"""
 os.makedirs(save_path, exist_ok=True)
 unet_model.save_pretrained(save_path)


print(f"\n{'='*80}")
print(f"TRAINING {config['name']} CONFIGURATION")
print(f"{'='*80}")
print(f"Max epochs: {num_epochs}")
print(f"Early stopping patience: {patience}")
print(f"Batch size: 4")
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Steps per epoch: {len(train_dataloader)}")
print(f"\nStarting training...\n")

training_start_time = time.time()
epoch = 0 # Initialize for final summary

for epoch in range(num_epochs):
 epoch_start_time = time.time()

 # Train
 train_loss = train_epoch(epoch)

 # Validate
 val_loss = validate()

 epoch_time = time.time() - epoch_start_time

 print(f"\nEpoch {epoch+1}/{num_epochs}:")
 print(f"Train Loss: {train_loss:.4f}")
 print(f"Val Loss: {val_loss:.4f}")
 print(f"Time: {epoch_time/60:.1f} min")
 print(f"LR: {optimizer.param_groups[0]['lr']:.2e}")

 # Save checkpoint every 5 epochs (LoRA only)
 if (epoch + 1) % 5 == 0:
 checkpoint_path = f"{LORA_SAVE_DIR}/lora_epoch_{epoch+1}"
 save_lora_weights(unet, checkpoint_path)
 print(f"Checkpoint saved: lora_epoch_{epoch+1}")

 # Early stopping check
 if val_loss < best_val_loss - min_delta:
 # Improvement found
 best_val_loss = val_loss
 patience_counter = 0

 # Save best model (LoRA only)
 best_path = f"{LORA_SAVE_DIR}/lora_best"
 save_lora_weights(unet, best_path)
 print(f"NEW BEST MODEL (val_loss: {val_loss:.4f})")
 else:
 # No improvement
 patience_counter += 1
 print(f"No improvement ({patience_counter}/{patience})")

 # Early stopping trigger
 if patience_counter >= patience:
 print(f"\n{'='*80}")
 print(f"EARLY STOPPING TRIGGERED")
 print(f"{'='*80}")
 print(f"Stopped at epoch {epoch+1}/{num_epochs}")
 print(f"Best validation loss: {best_val_loss:.4f}")
 print(f"No improvement for {patience} consecutive epochs")
 break

total_time = time.time() - training_start_time

print(f"\n{'='*80}")
print(f"TRAINING COMPLETE")
print(f"{'='*80}")
print(f"Total time: {total_time/60:.1f} minutes")
print(f"Epochs completed: {epoch+1}/{num_epochs}")
print(f"Best val loss: {best_val_loss:.4f}")
print(f"\nBest model: {LORA_SAVE_DIR}/lora_best")

print("Training with early stopping complete")


## Step 14: Save Final Model

In [ ]:
# Save final LoRA weights (LoRA only, not entire UNet)
final_path = f"{LORA_SAVE_DIR}/lora_final"
os.makedirs(final_path, exist_ok=True)
unet.save_pretrained(final_path)

print(f"\n Final LoRA weights saved: {final_path}")
print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print(f"\n{'='*80}")
print(f"Training complete for {config['name']} configuration!")
print(f"{'='*80}")
print(f"\nSaved checkpoints:")
print(f"lora_best/ - Best validation loss model")
print(f"lora_final/ - Final epoch model")
print(f"lora_epoch_X/ - Checkpoints every 5 epochs")
print(f"\nUse 'lora_best' for inference - it has the best LPIPS performance")
